# LeetCode #1285: Find the Right Interval

https://leetcode.com/problems/find-the-right-interval/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^2)$ | $O(n)$ |
| **Optimal: Binary Search on Sorted Starts ★** | $O(n \log n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
For each interval's end point, scan every other interval's start point to find the smallest one $\ge$ end. Quadratic in $n$.

### Optimal: Binary Search on Sorted Starts ★
Sort the start points while preserving original indices. For each interval's end point, binary search in the sorted starts to find the leftmost start $\ge$ end. Map back to the original index using the stored index alongside each start value.

**Why this is better than Brute Force:** Sorting is $O(n \log n)$ and each binary search is $O(\log n)$, giving $O(n \log n)$ total versus $O(n^2)$.

**Constraints:**
* $1 \le n \le 2 \times 10^4$
* $-10^6 \le start_i < end_i \le 10^6$
* All $start_i$ are distinct

## Solutions

### C#

In [ ]:
using System;
using System.Linq;
public class Solution {
    public int[] FindRightInterval(int[][] intervals) {
        int n = intervals.Length;
        // Pair each start point with its original index, then sort by start
        var starts = intervals.Select((iv, i) => (start: iv[0], idx: i))
                              .OrderBy(x => x.start).ToArray();
        var result = new int[n];
        for (int i = 0; i < n; i++) {
            int end = intervals[i][1];
            // Binary search for smallest start >= end
            int lo = 0, hi = n;
            while (lo < hi) {
                int mid = (lo + hi) / 2;
                if (starts[mid].start >= end) hi = mid;
                else lo = mid + 1;
            }
            // Map back to original index; -1 if no valid start exists
            result[i] = lo < n ? starts[lo].idx : -1;
        }
        return result;
    }
}

### Python

In [ ]:
import bisect
class Solution:
    def find_right_interval(self, intervals: list[list[int]]) -> list[int]:
        n = len(intervals)
        # Pair each start point with its original index, then sort by start
        starts = sorted((iv[0], i) for i, iv in enumerate(intervals))
        start_vals = [s for s, _ in starts]
        result = []
        for start, end in intervals:
            # Binary search for smallest start >= end
            pos = bisect.bisect_left(start_vals, end)
            # Map back to original index; -1 if no valid start exists
            result.append(starts[pos][1] if pos < n else -1)
        return result

### Go

In [ ]:
import "sort"
func findRightInterval(intervals [][]int) []int {
    n := len(intervals)
    type Pair struct{ start, idx int }
    // Pair each start point with its original index, then sort by start
    starts := make([]Pair, n)
    for i, iv := range intervals { starts[i] = Pair{iv[0], i} }
    sort.Slice(starts, func(i, j int) bool { return starts[i].start < starts[j].start })
    result := make([]int, n)
    for i, iv := range intervals {
        end := iv[1]
        // Binary search for smallest start >= end
        lo, hi := 0, n
        for lo < hi {
            mid := (lo + hi) / 2
            if starts[mid].start >= end { hi = mid } else { lo = mid + 1 }
        }
        // Map back to original index; -1 if no valid start exists
        if lo < n { result[i] = starts[lo].idx } else { result[i] = -1 }
    }
    return result
}

### Rust

In [ ]:
impl Solution {
    pub fn find_right_interval(intervals: Vec<Vec<i32>>) -> Vec<i32> {
        let n = intervals.len();
        // Pair each start point with its original index, then sort by start
        let mut starts: Vec<(i32, usize)> = intervals.iter().enumerate()
            .map(|(i, iv)| (iv[0], i)).collect();
        starts.sort_unstable();
        let start_vals: Vec<i32> = starts.iter().map(|&(s,_)| s).collect();
        intervals.iter().map(|iv| {
            let end = iv[1];
            // Binary search for smallest start >= end
            let pos = start_vals.partition_point(|&s| s < end);
            // Map back to original index; -1 if no valid start exists
            if pos < n { starts[pos].1 as i32 } else { -1 }
        }).collect()
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `intervals = [[1,2],[2,3],[3,4]]`
Sorted starts: [(1,0),(2,1),(3,2)]. End=2 → bisect finds start=2 at index 1 → original index 1. End=3 → start=3 → index 2. End=4 → no start ≥ 4 → -1. Answer: **[1, 2, -1]**.

### 2. Slightly Complex
**Input:** `intervals = [[3,4],[2,3],[1,2]]`
Sorted starts: [(1,2),(2,1),(3,0)]. End=4 → no start ≥ 4 → -1. End=3 → start=3 at idx 0 → result 0. End=2 → start=2 at idx 1 → result 1. Answer: **[-1, 0, 1]**.

### 3. Edge Case: Time Factor
**Input:** $n = 2 \times 10^4$ intervals.
Sorting is $O(n \log n)$ and $n$ binary searches each take $O(\log n)$ — total $O(n \log n) \approx 2 \times 10^4 \times 15 = 300{,}000$ operations.

### 4. Edge Case: Space Factor
**Input:** $n = 2 \times 10^4$ intervals.
The sorted starts array holds $n$ pairs — $O(n)$ extra space. No other auxiliary structure is needed.

### 5. Almost-Impossible but Plausible
**Input:** `intervals = [[-10^6, 10^6]]` (single interval spanning the entire range).
End = $10^6$; sorted starts = $[(-10^6, 0)]$. Binary search finds no start $\ge 10^6$. Answer: **[-1]**.